In [19]:
# =========================
# 🔌 MongoDB CONNECTION
# =========================

from pymongo import MongoClient

client = MongoClient(
    "mongodb://ich_editor:verystrongpassword"
    "@mongo.itcareerhub.de/?readPreference=primary"
    "&ssl=false&authMechanism=DEFAULT&authSource=ich_edit"
)

db = client["ich_edit"]
collection = db["final_project_dam130625_H_Ovsiannikova"]

print("Connected to MongoDB.")


# =========================
# 🗄️ MySQL CONNECTION (PyMySQL)
# =========================

import pymysql

conn = pymysql.connect(
    host='ich-db.edu.itcareerhub.de',
    port=3306,
    user='ich1',
    password='password',
    database='sakila',
    charset='utf8mb4'
)

cursor = conn.cursor()

try:
    cursor.execute("SELECT 1;")
    print("Connected to Sakila database.")
except Exception as e:
    print("Connection failed:", e)


# =========================
# 🔎 FUNCTIONS
# =========================

def show_categories(cursor):
    cursor.execute("SELECT category_id, name FROM category ORDER BY name;")
    rows = cursor.fetchall()
    print("\n=== Categories ===")
    for category_id, name in rows:
        print(f"{category_id}: {name}")
    print("==================\n")
    return rows


def show_year_range(cursor):
    cursor.execute("SELECT MIN(release_year), MAX(release_year) FROM film;")
    min_year, max_year = cursor.fetchone()
    print(f"\nFilm years: from {min_year} to {max_year}\n")
    return min_year, max_year


def search_by_keyword(cursor):
    keyword = input("Enter keyword: ")
    cursor.execute(
        "SELECT title, release_year FROM film WHERE title LIKE %s ORDER BY title;",
        (f"%{keyword}%",)
    )
    return cursor.fetchall()


def search_by_genre_and_year(cursor):
    category_id = input("Enter genre (number): ")
    year_from = input("Enter year from: ")
    year_to = input("Enter year to: ")

    query = """
    SELECT f.title, f.release_year, c.name
    FROM film AS f
    LEFT JOIN film_category AS fc ON f.film_id = fc.film_id
    LEFT JOIN category AS c ON fc.category_id = c.category_id
    WHERE c.category_id = %s
      AND f.release_year BETWEEN %s AND %s
    ORDER BY f.release_year;
    """

    cursor.execute(query, (category_id, year_from, year_to))
    return cursor.fetchall()


def show_paginated(results):
    index = 0
    total = len(results)

    if total == 0:
        print("No results found.")
        return

    while index < total:
        print("\n=== Results ===")
        for i in range(index, min(index + 10, total)):
            print(results[i])
        print("==============")

        index += 10
        if index < total:
            cont = input("Show next 10? (yes/no): ")
            if cont.lower() != 'yes':
                break


# =========================
# 🖥️ MAIN MENU
# =========================

def main():
    cursor = conn.cursor()

    show_categories(cursor)
    show_year_range(cursor)

    while True:
        print("\nSelect search type:")
        print("1 - Search by keyword")
        print("2 - Search by genre and year range")
        print("0 - Exit")

        choice = input("Your choice: ")

        if choice == "1":
            results = search_by_keyword(cursor)
            show_paginated(results)

        elif choice == "2":
            results = search_by_genre_and_year(cursor)
            show_paginated(results)

        elif choice == "0":
            break

        else:
            print("Invalid input, try again.")

    cursor.close()
    conn.close()
    print("Program exited.")


if __name__ == "__main__":
    main()

Connected to MongoDB.
Connected to Sakila database.

=== Categories ===
1: Action
2: Animation
3: Children
4: Classics
5: Comedy
6: Documentary
7: Drama
8: Family
9: Foreign
10: Games
11: Horror
12: Music
13: New
14: Sci-Fi
15: Sports
16: Travel


Film years: from 1990 to 2025


Select search type:
1 - Search by keyword
2 - Search by genre and year range
0 - Exit


Your choice:  1
Enter keyword:  EGG



=== Results ===
('AFRICAN EGG', 1998)
('EGG IGBY', 2009)
('RACER EGG', 2024)

Select search type:
1 - Search by keyword
2 - Search by genre and year range
0 - Exit


Your choice:  2
Enter genre (number):  5
Enter year from:  2010
Enter year to:  2015



=== Results ===
('FREEDOM CLEOPATRA', 2011, 'Comedy')
('LIFE TWISTED', 2011, 'Comedy')
('RUSHMORE MERMAID', 2012, 'Comedy')
('SUBMARINE BED', 2012, 'Comedy')
('HATE HANDICAP', 2013, 'Comedy')
('HUSTLER PARTY', 2013, 'Comedy')
('CAT CONEHEADS', 2014, 'Comedy')
('ZORRO ARK', 2014, 'Comedy')
('GOLD RIVER', 2015, 'Comedy')

Select search type:
1 - Search by keyword
2 - Search by genre and year range
0 - Exit


Your choice:  0


Program exited.
